# ⚡ NexusAI — Тримовне високопродуктивне навчання (УКР + РУС + АНГЛ, 100M - 6B) та KV-Cache інференс

Повнофункціональна студія для навчання та тестування моделей NexusAI:
- **BPE токенізатор на 16 000 токенів** з повною підтримкою української, російської та англійської мов (включаючи літери і, ї, є, ґ та апостроф).
- **Генератор датасетів на льоту (on-the-fly)**: процедурна генерація мільярдів токенів (математика, код, логіка, діалоги) прямо в оперативній пам'яті без необхідності зберігати гігабайти файлів.
- **Multi-GPU підтримка**: автоматичне використання 2x GPU (наприклад, 2x T4 в Kaggle) через PyTorch DataParallel.
- **KV-Cache прискорення**: швидка потокова генерація відповідей у чаті токен за токеном.


In [ ]:
import os, subprocess, sys, json, re, shutil, tempfile, time
from pathlib import Path

# ОПРЕДЕЛЕНИЕ И НАСТРОЙКА УСКОРИТЕЛЯ (GPU / TPU) НА KAGGLE
is_tpu = bool(os.environ.get('KAGGLE_TPU_NAME') or os.environ.get('TPU_NAME') or Path('/dev/accel0').exists())
if is_tpu:
    print('⚡ Обнаружен Cloud TPU VM на Kaggle. Настраиваем PJRT_DEVICE=TPU...')
    os.environ['PJRT_DEVICE'] = 'TPU'
else:
    print('🚀 Запуск в режиме GPU (NVIDIA CUDA). Будет задействован TF32 и нативный FlashAttention-2 / SDPA.')

REPO_URL = 'https://github.com/MrZombie121/nexusailoaderfiles.git'
WORK = Path('/kaggle/working/NexusAIprog')

if not WORK.exists():
    print(f'Клонирую репозиторий {REPO_URL}...')
    subprocess.run(['git', 'clone', REPO_URL, str(WORK)], check=True)
elif (WORK / '.git').exists():
    print('Обновляю репозиторий через git pull...')
    subprocess.run(['git', 'pull', '--ff-only'], cwd=WORK, check=True)

# Переход в директорию репозитория
os.chdir(str(WORK))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

print('Текущая папка:', Path.cwd())

if not (WORK / 'requirements.txt').exists():
    print('Содержимое /kaggle/working:', list(Path('/kaggle/working').glob('*')))
    raise FileNotFoundError(f'Не найден файл зависимостей: {WORK / "requirements.txt"}. Проверьте REPO_URL и успешность git clone.')

print('Устанавливаю необходимые зависимости...')
packages = ['-r', 'requirements.txt', 'gradio>=5.0', 'pandas', 'matplotlib', 'pyyaml', 'accelerate', 'transformers', '-e', '.']
if is_tpu:
    packages.append('torch_xla>=2.8.0')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, cwd=str(WORK), check=True)
print('Зависимости успешно установлены!')

DATA_DIR = WORK / 'datasets' / 'processed'
required = ['code.jsonl', 'math.jsonl', 'conversation.jsonl']
missing = [name for name in required if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Нет файлов данных в {DATA_DIR}: {missing}')
print('Данные готовы к обучению:', DATA_DIR)


## Проверка оборудования (GPU / TPU)
Проверяем доступность CUDA, объём VRAM, поддержку bfloat16, TF32 и FlashAttention-2.


In [ ]:
import torch

print('=== ПРОВЕРКА ОБОРУДОВАНИЯ KAGGLE ДЛЯ СКОРОСТНОГО ОБУЧЕНИЯ (100M-6B) ===')
if torch.cuda.is_available():
    gpu_cnt = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f'GPU: {gpu_cnt} x {gpu_name} ({vram_gb:.1f} GB VRAM на карту, суммарно {vram_gb * gpu_cnt:.1f} GB)')
    if gpu_cnt > 1:
        print(f'⚡ Multi-GPU: Обнаружено {gpu_cnt} видеокарт! Обучение будет автоматически распределено на все GPU через DataParallel.')
    print(f'Поддержка bfloat16: {"Да (максимальная скорость и точность)" if bf16_ok else "Нет (будет задействован FP16 с GradScaler)"}')
    
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')
    print('TensorFloat-32 (TF32): Включен (ускорение GEMM матриц до 3x)')
    print('FlashAttention / SDPA: Включен нативно в PyTorch 2.x')
else:
    try:
        import torch_xla.core.xla_model as xm
        device = xm.xla_device()
        print(f'TPU активен: {device}')
    except Exception:
        print('Внимание: Ни CUDA GPU, ни TPU не обнаружены. Обучение на CPU будет медленным!')


## Запуск визуальной панели NexusAI Studio (Kaggle)

Интерактивная среда с 4 специализированными разделами:
1. **Обучение / Дообучение**: выбор архитектуры (`100M` - `6B`), быстрый режим шагов (по умолчанию 200 шагов вместо десятков тысяч), настройка Micro Batch Size, Gradient Accumulation, Learning Rate, оптимизатора Adafactor/AdamW, живые графики Loss и Perplexity, автосохранение чекпойнтов.
2. **Интерактивный чат (Streaming)**: потоковый вывод токен за токеном с аппаратным **KV-Кэшем**, системным промптом, сэмплерами Min-P и Top-P. Модель кэшируется прямо в VRAM для мгновенных ответов.
3. **Playground**: генерация произвольных промптов с измерением скорости (токенов/сек).
4. **Аналитика**: расчёт Perplexity (PPL) на валидационных данных и пакетный тест контрольных вопросов.


In [ ]:
import sys, os
from pathlib import Path

# Автоматический поиск и гарантированное добавление корня проекта в sys.path
possible_roots = [
    Path('/kaggle/working/NexusAIprog'),
    Path('/content/NexusAIprog'),
    Path.cwd(),
    Path.cwd().parent,
]
WORK = next((p for p in possible_roots if (p / 'nexus').exists()), Path('/kaggle/working/NexusAIprog'))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
if (WORK / 'nexus').exists():
    try:
        os.chdir(str(WORK))
    except Exception:
        pass

import yaml, pandas as pd, gradio as gr, torch
import re, json, shutil, time

from nexus.tokenizer import SimpleTokenizer
from nexus.model import NexusModel
from nexus.generation import generate_stream, generate_text
from nexus.loss import compute_perplexity

MODEL_ROOT = Path('/kaggle/working/NexusAI/models')
CHECKPOINT_ROOT = Path('/kaggle/working/NexusAI/checkpoints')
VERSION_ROOT = Path('/kaggle/working/NexusAI/training_versions')
TOKENIZER_PATH = WORK / 'datasets' / 'tokenizer' / 'vocab.json'

STORAGE_PRESETS = {
    'Kaggle / NexusAI (Основная)': Path('/kaggle/working/NexusAI'),
    'Kaggle / Модели и чекпойнты': Path('/kaggle/working/outputs'),
    'Локально в проекте': WORK / 'outputs',
}

_MODEL_CACHE = {}

def get_cached_model(checkpoint_path):
    """Загружает модель с кэшированием в оперативной памяти/VRAM для мгновенных ответов."""
    checkpoint_path = str(Path(checkpoint_path).resolve())
    if checkpoint_path in _MODEL_CACHE:
        return _MODEL_CACHE[checkpoint_path]
    
    if not Path(checkpoint_path).exists():
        raise FileNotFoundError(f"Файл чекпойнта не найден: {checkpoint_path}")
        
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = checkpoint.get('config')
    if not config:
        raise ValueError('В чекпойнте отсутствует словарь config')
        
    local_vocab = Path(checkpoint_path).parent / 'vocab.json'
    vocab_file = local_vocab if local_vocab.exists() else TOKENIZER_PATH
    if not vocab_file.exists():
        vocab_file = WORK / 'datasets' / 'tokenizer' / 'vocab.json'
    tokenizer = SimpleTokenizer.load(vocab_file)
    
    m = config['model']
    model = NexusModel(
        vocab_size=tokenizer.vocab_size,
        hidden_size=int(m['hidden_size']),
        intermediate_size=int(m['intermediate_size']),
        num_layers=int(m['num_layers']),
        num_heads=int(m['num_heads']),
        num_key_value_heads=int(m.get('num_key_value_heads', m['num_heads'])),
        max_position_embeddings=int(m.get('max_position_embeddings', 2048)),
        dropout=0.0
    )
    raw_state = checkpoint.get('model_state', checkpoint)
    model.load_state_dict(raw_state)
    
    if device.type == 'cuda':
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model = model.to(dtype=dtype, device=device)
    else:
        model = model.to(device)
        
    model.eval()
    _MODEL_CACHE[checkpoint_path] = (model, tokenizer, device)
    return model, tokenizer, device

def storage_paths(place, custom_root, run_name):
    root = Path(custom_root).expanduser() if place == 'Своя папка' and str(custom_root).strip() else STORAGE_PRESETS.get(place, STORAGE_PRESETS['Kaggle / NexusAI (Основная)'])
    model_root, checkpoint_root, version_root = root / 'models', root / 'checkpoints', root / 'training_versions'
    summary = f'Модель: {model_root / run_name / "model.pt"}\nЧекпойнты: {checkpoint_root / run_name}\nВерсия: {version_root / run_name}'
    return model_root, checkpoint_root, version_root, summary

def show_storage(place, custom_root, run_name):
    return storage_paths(place, custom_root, str(run_name).strip() or 'nexus_run_01')[3]

def count_examples():
    return sum(1 for path in DATA_DIR.glob('*.jsonl') for _ in path.open(encoding='utf-8'))

def training_plan(model_size, mode, amount, batch_size=None, grad_accum=None, data_source=None):
    cfg = yaml.safe_load((WORK / 'configs' / f'{model_size}.yaml').read_text(encoding='utf-8'))
    bs = int(batch_size) if batch_size and int(batch_size) > 0 else int(cfg['training'].get('batch_size', 1))
    accum = int(grad_accum) if grad_accum and int(grad_accum) > 0 else int(cfg['training'].get('gradient_accumulation_steps', 1))
    is_synth = (data_source is None) or ('Генератор' in str(data_source)) or ('Синтетич' in str(data_source))
    examples = 1000000000 if is_synth else count_examples()
    batches_per_epoch = max(1, (examples + bs - 1) // bs)
    
    mode_str = str(mode).lower()
    if 'эпох' in mode_str:
        epochs = max(0.001, float(amount))
        batches = max(1, int(epochs * batches_per_epoch + 0.999999))
    else:
        batches = max(1, int(amount))
        epochs = batches / batches_per_epoch

    opt_steps = max(1, batches // accum)
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    gpu_speedup = max(1.0, num_gpus * 0.9)
    step_sec_map = {'100M': 0.015, '300M': 0.035, '1B': 0.12, '3B': 0.30, '6B': 0.75}
    est_sec = (batches * step_sec_map.get(model_size, 0.05)) / gpu_speedup
    if est_sec < 60:
        est_str = f'{est_sec:.0f} сек'
    elif est_sec < 3600:
        est_str = f'{est_sec/60:.1f} мин'
    else:
        est_str = f'{est_sec/3600:.2f} ч'
    return examples, bs, accum, batches_per_epoch, batches, opt_steps, epochs, est_str

def show_plan(model_size, mode, amount, max_length=256, batch_size=None, grad_accum=None, data_source=None):
    is_synth = data_source and ('Синтетическ' in str(data_source))
    examples, bs, accum, per_epoch, batches, opt_steps, epochs, est_time = training_plan(model_size, mode, amount, batch_size, grad_accum, data_source)
    eff_batch = bs * accum
    warning = ''
    if batches > 500:
        warning = ' [!] Внимание: выбрано более 500 шагов.'
    ex_str = 'бесконечно (генерация на лету, 1-10 млрд токенов)' if is_synth else str(examples)
    res = f'Примеров: {ex_str} | Micro-batch: {bs} | Accum: {accum}x (Эфф. батч: {eff_batch})\nДлина контекста: {max_length}\nШагов данных: {batches} (Шагов оптимизатора: {opt_steps}) | Эпох: {epochs:.3f}\nРасчетное время: ~{est_time}{warning}'
    return res

def make_config(model_size, max_steps, checkpoint_dir, checkpoint_every_epochs, resume, max_length=256, batch_size=None, grad_accum=None, lr=None, optimizer=None):
    base = yaml.safe_load((WORK / 'configs' / f'{model_size}.yaml').read_text(encoding='utf-8'))
    base['training']['max_steps'] = int(max_steps)
    base['training'].pop('epochs', None)
    base['training']['checkpoint_dir'] = str(checkpoint_dir)
    base['training']['checkpoint_every_epochs'] = max(1, int(checkpoint_every_epochs))
    base['training']['max_seq_length'] = int(max_length)
    if batch_size is not None and int(batch_size) > 0:
        base['training']['batch_size'] = int(batch_size)
    if grad_accum is not None and int(grad_accum) > 0:
        base['training']['gradient_accumulation_steps'] = int(grad_accum)
    if lr is not None and float(lr) > 0:
        base['training']['learning_rate'] = float(lr)
    if optimizer and str(optimizer).strip():
        base['training']['optimizer'] = str(optimizer).strip()
    if resume and resume.strip():
        base['training']['resume_from'] = resume.strip()
    path = Path('/tmp/nexus_kaggle_config.yaml')
    path.write_text(yaml.safe_dump(base, allow_unicode=True), encoding='utf-8')
    return path

def package_version(run_name, checkpoint_dir, version_root, model_root, config_path):
    latest = checkpoint_dir / 'latest.pt'
    if not latest.exists():
        latest = checkpoint_dir / 'model.pt'
    if not latest.exists():
        return 'Чекпойнт latest.pt не найден'
    
    config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    readme_content = (
        f"# NexusAI Model: {run_name}\n\n"
        f"## Параметры обучения\n"
        f"- Конфигурация модели: {config.get('model', {})}\n"
        f"- Шагов (max_steps): {config['training'].get('max_steps')}\n"
        f"- Batch size: {config['training'].get('batch_size')}\n"
        f"- Gradient Accumulation: {config['training'].get('gradient_accumulation_steps', 1)}\n"
        f"- Learning Rate: {config['training'].get('learning_rate')}\n"
        f"- Длина контекста: {config['training'].get('max_seq_length', 256)}\n"
        f"- Дата упаковки: {time.strftime('%Y-%m-%d %H:%M:%S')}\n"
    )

    model_target = model_root / run_name
    version_target = version_root / run_name
    model_target.mkdir(parents=True, exist_ok=True)
    version_target.mkdir(parents=True, exist_ok=True)
    
    shutil.copy2(latest, model_target / 'model.pt')
    shutil.copy2(latest, version_target / 'model.pt')
    shutil.copy2(config_path, model_target / 'config.yaml')
    shutil.copy2(config_path, version_target / 'config.yaml')
    
    (model_target / 'README.txt').write_text(readme_content, encoding='utf-8')
    (version_target / 'README.txt').write_text(readme_content, encoding='utf-8')

    if TOKENIZER_PATH.exists():
        shutil.copy2(TOKENIZER_PATH, model_target / 'vocab.json')
        shutil.copy2(TOKENIZER_PATH, version_target / 'vocab.json')
        spm_model = TOKENIZER_PATH.parent / 'tokenizer.model'
        if spm_model.exists():
            shutil.copy2(spm_model, model_target / 'tokenizer.model')
            shutil.copy2(spm_model, version_target / 'tokenizer.model')
    (version_target / 'manifest.json').write_text(
        json.dumps({'run_name': run_name, 'checkpoint': str(latest), 'model': str(model_target / 'model.pt')}, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )
    return f'Модель сохранена:\n{model_target}'

def package_existing(run_name, place, custom_root):
    run_name = re.sub(r'[^a-zA-Z0-9_-]+', '_', str(run_name).strip()) or 'run'
    model_root, checkpoint_root, version_root, _ = storage_paths(place, custom_root, run_name)
    checkpoint_dir = Path(checkpoint_root) / run_name
    latest = checkpoint_dir / 'model.pt' if (checkpoint_dir / 'model.pt').exists() else checkpoint_dir / 'latest.pt'
    if not latest.exists():
        return f'Не найден чекпойнт в {checkpoint_dir}'
    state = torch.load(latest, map_location='cpu')
    config_path = Path('/tmp/nexus_existing_config.yaml')
    config_path.write_text(yaml.safe_dump(state.get('config', {}), allow_unicode=True), encoding='utf-8')
    return package_version(run_name, checkpoint_dir, Path(version_root), Path(model_root), config_path)

def train_stream(model_size, mode, amount, checkpoint_every_epochs, run_name, place, custom_root, resume, max_length=256, batch_size=None, grad_accum=None, lr=None, optimizer=None, data_source=None):
    is_synth = (data_source is None) or ('Генератор' in str(data_source)) or ('Синтетич' in str(data_source))
    if not is_synth and not any(DATA_DIR.glob('*.jsonl')):
        yield 'Нет JSONL-файлов в datasets/processed', pd.DataFrame(columns=['step','loss','ppl','lr'])
        return

    run_name = re.sub(r'[^a-zA-Z0-9_-]+', '_', str(run_name).strip()) or 'run'
    model_root, checkpoint_root, version_root, _ = storage_paths(place, custom_root, run_name)
    examples, bs, accum, per_epoch, max_steps, opt_steps, planned_epochs, est_time = training_plan(model_size, mode, amount, batch_size, grad_accum, data_source)
    log_prefix = (
        f"🚀 Старт: {examples} примеров | batch_size={bs} | accum={accum}x | контекст={max_length} | "
        f"шагов={max_steps} (~{planned_epochs:.3f} эпох) | расчетное время: ~{est_time}"
    )

    checkpoint_dir = Path(checkpoint_root) / run_name
    config_path = make_config(model_size, max_steps, checkpoint_dir, checkpoint_every_epochs, resume, max_length, bs, accum, lr, optimizer)

    command = [
        sys.executable, 'train.py',
        '--config', str(config_path),
        '--data', str(DATA_DIR),
        '--max-length', str(max_length),
        '--max-steps', str(max_steps)
    ]
    if is_synth:
        command.append('--synthetic')
    if resume and resume.strip():
        command.extend(['--resume', resume.strip()])

    process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    rows, log_lines = [], [log_prefix]

    for line in process.stdout:
        clean_line = line.rstrip()
        log_lines.append(clean_line)
        match = re.search(r'step=(\d+).*?loss=([0-9.eE+-]+).*?(?:ppl=([0-9.eE+-]+))?.*?lr=([0-9.eE+-]+)', clean_line)
        if match:
            step_val = int(match.group(1))
            loss_val = float(match.group(2))
            ppl_val = float(match.group(3)) if match.group(3) else compute_perplexity(loss_val)
            lr_val = float(match.group(4))
            rows.append({'step': step_val, 'loss': loss_val, 'ppl': ppl_val, 'lr': lr_val})
        yield '\n'.join(log_lines[-35:]), pd.DataFrame(rows)

    code = process.wait()
    log_lines.append(f'--- ОБУЧЕНИЕ ЗАВЕРШЕНО С КОДОМ: {code} ---')
    log_lines.append(f'Выполнено: {max_steps} шагов данных (~{planned_epochs:.3f} эпох)')

    packaging_result = package_version(run_name, checkpoint_dir, Path(version_root), Path(model_root), config_path)
    log_lines.append(f'Результат упаковки модели: {packaging_result}')

    yield '\n'.join(log_lines[-35:]), pd.DataFrame(rows)

def chat_stream(message, history, checkpoint_path, system_prompt, temperature, top_p, min_p, repetition_penalty, max_new_tokens):
    """Потоковая генерация ответа токен за токеном с использованием KV-кэша и современных сэмплеров."""
    if not checkpoint_path or not Path(checkpoint_path).exists():
        yield "⚠️ Ошибка: укажите существующий путь к чекпойнту (например, latest.pt или model.pt)"
        return

    try:
        model, tokenizer, device = get_cached_model(checkpoint_path)
    except Exception as e:
        yield f"⚠️ Ошибка загрузки модели: {e}"
        return

    prompt_lines = []
    if system_prompt and system_prompt.strip():
        prompt_lines.append(f"Інструкція / Instruction: {system_prompt.strip()}")

    for item in history:
        if isinstance(item, (list, tuple)) and len(item) == 2:
            u, b = item
            if u: prompt_lines.append(f"Пользователь: {u}")
            if b: prompt_lines.append(f"Ассистент: {b}")
        elif isinstance(item, dict):
            role = "Пользователь" if item.get("role") == "user" else "Ассистент"
            prompt_lines.append(f"{role}: {item.get('content', '')}")

    prompt_lines.append(f"Пользователь: {message}\nАссистент:")
    full_prompt = "\n".join(prompt_lines)

    for chunk in generate_stream(
        model=model,
        tokenizer=tokenizer,
        prompt=full_prompt,
        max_new_tokens=int(max_new_tokens),
        temperature=float(temperature),
        top_k=50,
        top_p=float(top_p),
        min_p=float(min_p),
        repetition_penalty=float(repetition_penalty),
        do_sample=(float(temperature) > 0.0),
        use_cache=True,
        stop_strings=["\nПользователь:", "\nUser:", "\nHuman:", "\n###", "<eos>"],
        device=device,
    ):
        yield chunk

def playground_generate(prompt, checkpoint_path, system_prompt, temperature, top_p, min_p, repetition_penalty, max_new_tokens):
    if not checkpoint_path or not Path(checkpoint_path).exists():
        return "⚠️ Чекпойнт не найден. Укажите существующий путь."
    try:
        model, tokenizer, device = get_cached_model(checkpoint_path)
    except Exception as e:
        return f"⚠️ Ошибка загрузки модели: {e}"

    full_prompt = f"Инструкция: {system_prompt.strip()}\n{prompt}" if system_prompt and system_prompt.strip() else prompt

    t0 = time.time()
    result = generate_text(
        model=model,
        tokenizer=tokenizer,
        prompt=full_prompt,
        max_new_tokens=int(max_new_tokens),
        temperature=float(temperature),
        top_k=50,
        top_p=float(top_p),
        min_p=float(min_p),
        repetition_penalty=float(repetition_penalty),
        do_sample=(float(temperature) > 0.0),
        use_cache=True,
        stop_strings=["\nПользователь:", "\nUser:", "\n###", "<eos>"],
        return_full_text=False,
        device=device,
    )
    elapsed = max(time.time() - t0, 1e-4)
    tokens_generated = len(tokenizer.encode(result))
    speed = tokens_generated / elapsed
    return f"{result}\n\n[Сгенерировано {tokens_generated} токенов за {elapsed:.2f} сек ({speed:.1f} токенов/сек)]"

def calculate_perplexity(checkpoint_path, val_data_path):
    import math
    if not Path(checkpoint_path).exists():
        return "Чекпойнт не найден"
    try:
        model, tokenizer, device = get_cached_model(checkpoint_path)
    except Exception as e:
        return f"Ошибка: {e}"

    from nexus.data_loader import get_data_loader
    val_path = Path(val_data_path)
    if not val_path.exists():
        return f"Путь {val_path} не существует"

    loader = get_data_loader(val_path, batch_size=4, max_length=128)
    total_loss, total_count = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch[0].to(device)
            targets = batch[1].to(device)
            logits = model(input_ids)
            loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
            total_loss += loss.item()
            total_count += 1
            if total_count >= 100:
                break

    avg_loss = total_loss / max(1, total_count)
    ppl = compute_perplexity(avg_loss)
    return f"Perplexity (PPL): {ppl:.2f} | Cross-Entropy Loss: {avg_loss:.4f} (на {total_count} батчах)"

def evaluate_batch(checkpoint_path, prompts):
    if not Path(checkpoint_path).exists():
        return pd.DataFrame([{"prompt": "Ошибка", "answer": f"Чекпойнт не найден: {checkpoint_path}"}])
    try:
        model, tokenizer, device = get_cached_model(checkpoint_path)
    except Exception as e:
        return pd.DataFrame([{"prompt": "Ошибка", "answer": str(e)}])

    rows = []
    for line in prompts.splitlines():
        line = line.strip()
        if not line:
            continue
        formatted_prompt = f"Пользователь: {line}\nАссистент:"
        answer = generate_text(
            model=model,
            tokenizer=tokenizer,
            prompt=formatted_prompt,
            max_new_tokens=128,
            temperature=0.7,
            top_p=0.9,
            min_p=0.05,
            repetition_penalty=1.15,
            use_cache=True,
            stop_strings=["\nПользователь:", "\nUser:", "\n###", "<eos>"],
            return_full_text=False,
            device=device,
        )
        rows.append({'prompt': line, 'answer': answer})
    return pd.DataFrame(rows)


In [ ]:
with gr.Blocks(title='NexusAI Studio (Kaggle)') as app:
    gr.Markdown(
        "# ⚡ NexusAI Studio — Высокопроизводительное обучение (100M - 6B) и KV-Cache генерация (Kaggle)\n\n"
        "Включены передовые технологии: **FlashAttention-2 / SDPA**, **KV-Cache Decoding**, **bfloat16 AMP**, "
        "**Fused AdamW / Adafactor**, **Decoupled Weight Decay**, **Min-P & Top-P Sampling**, **Streaming Output**."
    )

    with gr.Tab('🚀 Обучение / Дообучение'):
        with gr.Row():
            size = gr.Dropdown(['100M', '300M', '1B', '3B', '6B'], value='300M', label='Архитектура модели')
            max_length = gr.Dropdown([128, 256, 512, 1024, 2048], value=256, label='Контекст (256 рек. для 1B-6B, ускорение 2-4x)')
            data_source = gr.Dropdown(['Генератор на льоту (УКР + РУС + АНГЛ, безліміт)', 'Локальні файли (datasets/processed)'], value='Генератор на льоту (УКР + РУС + АНГЛ, безліміт)', label='Джерело даних')
            mode = gr.Radio(['Фиксированные шаги (Max Steps)', 'По эпохам'], value='Фиксированные шаги (Max Steps)', label='Режим задания лимита')
            amount = gr.Number(value=200, minimum=1, precision=0, label='Количество шагов (Max Steps)')

        with gr.Accordion('⚙️ Расширенные гиперпараметры обучения', open=False):
            with gr.Row():
                batch_size = gr.Number(value=32, minimum=1, precision=0, label='Micro Batch Size (на устройство)')
                grad_accum = gr.Number(value=2, minimum=1, precision=0, label='Gradient Accumulation Steps')
                lr = gr.Number(value=0.0004, minimum=1e-6, label='Learning Rate')
                optimizer_choice = gr.Dropdown(['adafactor', 'adamw'], value='adafactor', label='Оптимизатор')

        with gr.Row():
            checkpoint_every_epochs = gr.Number(value=1, minimum=1, precision=0, label='Сохранять чекпойнт каждые N эпох')
            run_name = gr.Textbox(value='nexus_run_01', label='Имя версии / эксперимента')

        with gr.Row():
            place = gr.Dropdown(['Kaggle / NexusAI (Основная)', 'Kaggle / Модели и чекпойнты', 'Локально в проекте', 'Своя папка'], value='Kaggle / NexusAI (Основная)', label='Куда сохранять')
            custom_root = gr.Textbox(value='/kaggle/working/NexusAI_Custom', label='Своя папка (для варианта «Своя папка»)')

        storage_info = gr.Textbox(label='Целевые пути сохранения', lines=3, interactive=False)

        with gr.Row():
            plan_button = gr.Button('📊 Рассчитать план обучения')
            train_button = gr.Button('▶️ Запустить обучение', variant='primary')

        plan = gr.Textbox(label='Детальный план обучения', lines=6, interactive=False)
        resume = gr.Textbox(label='Чекпойнт для дообучения (путь к .pt)', placeholder='Оставьте пустым для обучения с нуля')

        log = gr.Textbox(label='Журнал обучения (Live)', lines=12)
        loss_plot = gr.LinePlot(x='step', y='loss', title='Кривая Loss по шагам', y_title='Loss', x_title='Шаг оптимизатора')

        with gr.Row():
            package_button = gr.Button('📦 Упаковать последнюю модель из чекпойнта')
            package_status = gr.Textbox(label='Результат упаковки', lines=3, interactive=False)

        plan_button.click(show_plan, [size, mode, amount, max_length, batch_size, grad_accum, data_source], plan)
        for component in [place, custom_root, run_name]:
            component.change(show_storage, [place, custom_root, run_name], storage_info)
        app.load(show_storage, [place, custom_root, run_name], storage_info)

        train_button.click(
            train_stream,
            [size, mode, amount, checkpoint_every_epochs, run_name, place, custom_root, resume, max_length, batch_size, grad_accum, lr, optimizer_choice, data_source],
            [log, loss_plot]
        )
        package_button.click(package_existing, [run_name, place, custom_root], package_status)

    with gr.Tab('💬 Интерактивный чат (Streaming)'):
        with gr.Row():
            chat_checkpoint = gr.Textbox(
                value=str(MODEL_ROOT / 'nexus_run_01' / 'model.pt'),
                label='Путь к модели (.pt)',
                scale=4
            )
            refresh_btn = gr.Button('🔄 Перезагрузить веса', scale=1)

        system_prompt = gr.Textbox(
            value='Ти корисний тримовний помічник NexusAI (вільно володієш українською, російською та англійською мовами). Відповідай чітко, структуровано та ввічливо.',
            label='Системная инструкция (System Prompt)',
            lines=2
        )

        with gr.Row():
            temperature = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='Temperature (0 = greedy)')
            top_p = gr.Slider(0.1, 1.0, value=0.9, step=0.05, label='Top-P (Nucleus)')
            min_p = gr.Slider(0.0, 0.5, value=0.05, step=0.01, label='Min-P (Динамический порог)')
            repetition_penalty = gr.Slider(1.0, 2.0, value=1.15, step=0.05, label='Repetition Penalty')
            max_new_tokens = gr.Slider(16, 1024, value=256, step=16, label='Max New Tokens')

        chatbot = gr.Chatbot(label='Диалог с NexusAI', height=450)
        chat_msg = gr.Textbox(placeholder='Введите сообщение на русском или английском...', label='Ваше сообщение')
        with gr.Row():
            send_btn = gr.Button('Отправить', variant='primary')
            clear_btn = gr.Button('Очистить диалог')

        def user_turn(user_message, history):
            return "", history + [[user_message, None]]

        def bot_turn(history, checkpoint_path, sys_prompt, temp, p_val, min_p_val, rep_pen, max_tok):
            user_msg = history[-1][0]
            prior_history = history[:-1]
            history[-1][1] = ""
            for partial in chat_stream(user_msg, prior_history, checkpoint_path, sys_prompt, temp, p_val, min_p_val, rep_pen, max_tok):
                history[-1][1] = partial
                yield history

        chat_msg.submit(user_turn, [chat_msg, chatbot], [chat_msg, chatbot], queue=False).then(
            bot_turn, [chatbot, chat_checkpoint, system_prompt, temperature, top_p, min_p, repetition_penalty, max_new_tokens], chatbot
        )
        send_btn.click(user_turn, [chat_msg, chatbot], [chat_msg, chatbot], queue=False).then(
            bot_turn, [chatbot, chat_checkpoint, system_prompt, temperature, top_p, min_p, repetition_penalty, max_new_tokens], chatbot
        )
        clear_btn.click(lambda: None, None, chatbot, queue=False)
        refresh_btn.click(lambda path: _MODEL_CACHE.pop(str(Path(path).resolve()), None) and 'Сброшено' or 'Сброшено', [chat_checkpoint], None)

    with gr.Tab('🧪 Playground (Продвинутая генерация)'):
        with gr.Row():
            pg_checkpoint = gr.Textbox(value=str(MODEL_ROOT / 'nexus_run_01' / 'model.pt'), label='Путь к модели')
        with gr.Row():
            pg_system = gr.Textbox(value='', label='Системный контекст (опционально)', lines=2)
        with gr.Row():
            pg_prompt = gr.Textbox(lines=5, label='Входной промпт', placeholder='Задача: Реши уравнение 2x + 5 = 15...')
        with gr.Row():
            pg_temp = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='Temperature')
            pg_top_p = gr.Slider(0.1, 1.0, value=0.9, step=0.05, label='Top-P')
            pg_min_p = gr.Slider(0.0, 0.5, value=0.05, step=0.01, label='Min-P')
            pg_rep = gr.Slider(1.0, 2.0, value=1.15, step=0.05, label='Repetition Penalty')
            pg_tokens = gr.Slider(16, 1024, value=256, step=16, label='Max New Tokens')

        pg_button = gr.Button('Сгенерировать текст', variant='primary')
        pg_output = gr.Textbox(label='Сгенерированный ответ и метрики скорости', lines=10)
        pg_button.click(playground_generate, [pg_prompt, pg_checkpoint, pg_system, pg_temp, pg_top_p, pg_min_p, pg_rep, pg_tokens], pg_output)

    with gr.Tab('📈 Аналитика и бенчмарки'):
        with gr.Row():
            eval_checkpoint = gr.Textbox(value=str(MODEL_ROOT / 'nexus_run_01' / 'model.pt'), label='Путь к модели')
            val_path = gr.Textbox(value=str(DATA_DIR), label='Путь к данным для валидации')
        ppl_button = gr.Button('Рассчитать Perplexity (PPL)')
        ppl_output = gr.Textbox(label='Результат оценки Perplexity')
        ppl_button.click(calculate_perplexity, [eval_checkpoint, val_path], ppl_output)

        gr.Markdown('### Пакетное тестирование на контрольных вопросах')
        default_prompts = (
            "Привіт! Розкажи про себе.\n"
            "Розв'яжи рівняння: 3x + 15 = 60.\n"
            "Напиши функцію на Python для обчислення чисел Фібоначчі.\n"
            "What is 25 multiplied by 16?\n"
            "Привет! В чём разница между TCP и UDP?"
        )
        test_prompts = gr.Textbox(lines=6, value=default_prompts, label='Контрольные промпты (по одному на строку)')
        batch_eval_btn = gr.Button('Запустить пакетный тест')
        batch_results = gr.Dataframe(headers=['prompt', 'answer'], interactive=False)
        batch_eval_btn.click(evaluate_batch, [eval_checkpoint, test_prompts], batch_results)

print('Gradio запускается. Если публичная ссылка не появилась, смотрите вывод ниже.')
app.queue().launch(share=True, debug=True, show_error=True)


## Як користуватися блокнотом на Kaggle

1. **Налаштування прискорювача**:
   - У правій панелі Kaggle (**Settings → Accelerator**) оберіть **GPU T4 x 2** (2x NVIDIA Tesla T4) або **GPU P100**.
   - При виборі 2 GPU скрипт автоматично використовує обидві відеокарти через **Multi-GPU DataParallel** для 2x прискорення.
   - Переконайтеся, що перемикач **Internet on** увімкнений.

2. **Тримовний BPE токенізатор (16 000 токенів)**:
   - Повноцінно розпізнає українську, російську та англійську мови (літери і, ї, є, ґ, апостроф), математику та код.

3. **Навчання з процедурним генератором на льоту**:
   - За замовчуванням у вкладці **Обучение / Дообучение** увімкнено джерело: **«Генератор на льоту (УКР + РУС + АНГЛ, безліміт)»**.
   - Модель навчається на нескінченному потоці прикладів математики, коду та діалогів, без необхідності завантажувати важкі файли.
   - Початковий ліміт встановлено на **200 кроків** (завершується менш ніж за 1 хвилину).

4. **Інтерактивний чат**:
   - У вкладці **Інтерактивний чат (Streaming)** спілкуйтеся з моделлю будь-якою з 3 мов з підтримкою швидкого KV-кэшу.
